### Load All Components

In [2]:
# Load All Components ───────────
import faiss
import numpy as np
import json
import os
import anthropic
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# Paths ──────────────────────────────────
CHUNKS_FILE = '../data/processed/chunks.json'
FAISS_FILE  = '../vector_store/faiss_index/alzheimer.index'

# Load chunks ────────────────────────────
print(" Loading components...")
with open(CHUNKS_FILE) as f:
    chunks = json.load(f)
print(f" Chunks:  {len(chunks):,}")

# Load FAISS ─────────────────────────────
index = faiss.read_index(FAISS_FILE)
print(f" FAISS:   {index.ntotal:,} vectors")

# Load Model ─────────────────────────────
print(f" Loading PubMedBERT...")
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print(f" Model:   PubMedBERT 768-dim")

# Load Claude ────────────────────────────
client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)
print(f" Claude:  API ready!")
print(f"\n All components loaded!")

 Loading components...
 Chunks:  40,995
 FAISS:   40,995 vectors
 Loading PubMedBERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 593.67it/s, Materializing param=pooler.dense.weight]                               


 Model:   PubMedBERT 768-dim
 Claude:  API ready!

 All components loaded!


###  Retrieve Function

In [3]:
# Retrieve Chunks ────────────────
def retrieve_chunks(query, k=5):
    """
    Find most relevant chunks
    using FAISS semantic search
    """
    # Encode query
    query_vector = model.encode([query])
    query_vector = query_vector.astype('float32')

    # Search FAISS
    distances, indices = index.search(
        query_vector, k=k
    )

    # Build results
    results = []
    for dist, idx in zip(
        distances[0], indices[0]
    ):
        chunk = chunks[idx]
        results.append({
            "title":    chunk['title'],
            "authors":  chunk['authors'],
            "year":     chunk['year'],
            "journal":  chunk['journal'],
            "chunk":    chunk['chunk'],
            "pmid":     chunk['pmid'],
            "distance": float(dist)
        })

    return results

print("✅ Retrieve function ready!")

✅ Retrieve function ready!


### Claude Answer Function

In [4]:
### Generate Answer ────────────────
def generate_answer(query, retrieved):
    """
    Generate cited answer using Claude
    """
    # Build context from retrieved chunks
    context = ""
    for i, r in enumerate(retrieved):
        context += f"""
Source {i+1}:
Title:   {r['title']}
Authors: {r['authors']}
Year:    {r['year']}
Journal: {r['journal']}
Content: {r['chunk']}
---
"""

    # Build prompt
    prompt = f"""You are AlzDetect AI — an expert 
medical research assistant specializing in 
Alzheimer's disease research.

Answer the question using ONLY the provided 
PubMed research sources below.

Rules:
1. Cite sources as (Source 1), (Source 2) etc
2. Be specific — mention biomarker names,
   percentages, findings when available
3. If sources don't fully answer — say so
4. Never hallucinate or add outside knowledge
5. End with "Sources Used" section

RESEARCH SOURCES:
{context}

QUESTION: {query}

Provide a comprehensive, cited answer:"""

    # Call Claude
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )

    return response.content[0].text

print("✅ Generate function ready!")

✅ Generate function ready!


### Main RAG Function

In [ ]:
### Main RAG Function ──────────────
def ask_alzdetect(query):
    """
    Full RAG pipeline:
    Query → Retrieve → Generate → Answer
    """
    print(f"\n{'='*55}")
    print(f" QUESTION: {query}")
    print(f"{'='*55}")

    # Step 1: Retrieve
    print(f" Searching 40,995 chunks...")
    retrieved = retrieve_chunks(query, k=5)
    print(f" Found {len(retrieved)} relevant chunks!")

    # Show sources found
    print(f"\n Sources retrieved:")
    for i, r in enumerate(retrieved):
        print(f"   {i+1}. {r['title'][:55]}...")
        print(f"      Year: {r['year']} | Distance: {r['distance']:.2f}")

    # Step 2: Generate
    print(f"\n Generating answer with Claude...")
    answer = generate_answer(query, retrieved)

    # Step 3: Display
    print(f"\n{'='*55}")
    print(f"🤖 ANSWER:")
    print(f"{'='*55}")
    print(answer)
    print(f"{'='*55}")

    return answer

print(" ask_alzdetect() ready!")
print("\n RAG Pipeline complete!")


 ask_alzdetect() ready!

 RAG Pipeline complete!
   Run Cell 5 to test!


In [7]:
## First Real Test ────────────────
answer1 = ask_alzdetect(
    "What blood biomarkers can detect "
    "Alzheimer's disease early?"
)


 QUESTION: What blood biomarkers can detect Alzheimer's disease early?
 Searching 40,995 chunks...
 Found 5 relevant chunks!

 Sources retrieved:
   1. Evidence to Consider Angiotensin II Receptor Blockers f...
      Year: 2016 | Distance: 90.41
   2. A possible blood plasma biomarker for early-stage Alzhe...
      Year: 2022 | Distance: 105.87
   3. Validating blood tests as a possible routine diagnostic...
      Year: 2023 | Distance: 106.05
   4. Plasma p-tau181, p-tau217, and other blood-based Alzhei...
      Year: 2021 | Distance: 107.80
   5. Developing non-invasive molecular markers for early ris...
      Year: 2025 | Distance: 110.21

 Generating answer with Claude...

🤖 ANSWER:
# Blood Biomarkers for Early Alzheimer's Disease Detection

Based on the provided research sources, here is what we know about blood biomarkers for early Alzheimer's disease (AD) detection:

## Specific Biomarkers Identified

**Plasma p-tau181 and p-tau217** are blood-based AD biomarkers that have been